***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 2 章：干涉测量的数学工具箱](2_0_introduction.ipynb)
    * 上一节：[2.13 补充专题：球面三角学](2_13_spherical_trigonometry.ipynb)
    * 下一节：[2.x 延伸阅读与参考文献](2_x_further_reading_and_references.ipynb)

***


导入标准模块。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
%matplotlib inline
HTML('../style/course.css')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False


## 2.14 补充专题：一维 CLEAN 的数学演示<a id='math:sec:clean_1d'></a><!--\label{math:sec:clean_1d}-->

CLEAN 通常被放在成像与去卷积章节中讨论，但它所依赖的数学并不神秘：傅里叶变换说明可见度与天空亮度之间的关系，采样函数说明为什么观测信息不完整，卷积定理说明脏图像为什么带有旁瓣，线性代数和最小二乘则说明去卷积本质上是在解一个病态反问题。本节把这些工具放到一个一维模型中，提前展示 CLEAN 的数学骨架。

这里的目标不是替代第 6 章的成像算法讨论，而是建立一条清楚的因果链。只要看清楚“采样如何产生脏波束、脏波束如何污染图像、CLEAN 如何逐步建立点源模型”，后续二维成像中的自然加权、均匀加权、主循环、次循环和恢复波束都会更容易理解。


### 2.14.1 从测量方程到正规方程<a id='math:sec:clean_normal_equation'></a>

把天空亮度离散成向量 $I$，把傅里叶变换写成线性算子 $F$，把实际观测到的 `uv` 采样写成选择算子 $S$，观测方程可以写成

$$
V_{\rm obs}=SFI+n,
$$

其中 $V_{\rm obs}$ 是观测可见度，$n$ 是热噪声、校准残差和模型误差的合并记号。若暂时把噪声视为均值为零的高斯噪声，则加权最小二乘目标函数为

$$
\chi^2(I)=(SFI-V_{\rm obs})^H W (SFI-V_{\rm obs}),
$$

其中 $W$ 是权重矩阵。统计意义上，$W$ 可以取噪声协方差矩阵的逆；成像实践中，$W$ 还会被用来调节分辨率、旁瓣和灵敏度之间的平衡。因此权重既有统计含义，也有成像策略含义。

令 $M=SF$，目标函数就是 $(MI-V_{\rm obs})^H W(MI-V_{\rm obs})$。对 $I$ 求导并令梯度为零，可得正规方程

$$
M^HWM I = M^H W V_{\rm obs}.
$$

把 $M=SF$ 代回去，得到

$$
F^H S^H W S F I = F^H S^H W V_{\rm obs}.
$$

右端是由加权可见度反变换得到的图像，通常称为脏图像，记为

$$
I_D = F^H S^H W V_{\rm obs}.
$$

左端的算子

$$
H = F^H S^H W S F
$$

就是这个最小二乘问题的 Hessian。在平移不变的一维或窄视场二维近似下，$H$ 的作用可以理解为一次卷积；卷积核正是由加权采样函数反傅里叶变换得到的脏波束或 PSF。因此正规方程可以写成非常紧凑的形式

$$
H I = I_D.
$$

这条式子是 CLEAN 的数学入口。干涉阵列没有测到所有傅里叶分量，$S$ 中存在大量空洞，$H$ 往往奇异或严重病态。未采样的空间频率无法从数据中凭空恢复，直接求逆会把噪声和旁瓣结构一同放大。去卷积算法必须引入额外假设：CLEAN 采用的核心假设是，天空中相当一部分亮度可以由一组紧致分量近似表示。


### 2.14.2 脏图像为什么是卷积<a id='math:sec:dirty_image_convolution'></a>

为了把问题写得足够透明，先考虑一维天空坐标 $l$ 与一维空间频率 $u$。若主波束暂时取为常数，理想可见度为

$$
V(u)=\int I(l)e^{-2\pi iul}\,dl.
$$

实际观测只在有限的 $u$ 位置上取得样本，并为这些样本赋予权重 $W(u)$。于是反变换得到的脏图像为

$$
I_D(l)=\int W(u)S(u)V(u)e^{2\pi iul}\,du.
$$

将 $V(u)$ 的定义代入，交换积分次序，有

$$
\begin{aligned}
I_D(l)
&=\int I(l')\left[\int W(u)S(u)e^{2\pi iu(l-l')}\,du\right]dl'\\
&=\int I(l')B_D(l-l')\,dl'\\
&=(I*B_D)(l),
\end{aligned}
$$

其中

$$
B_D(l)=\int W(u)S(u)e^{2\pi iul}\,du
$$

就是脏波束。这个推导说明，脏图像不是“有噪声的真图像”这么简单，而是真天空经过脏波束卷积后的结果。点源不会只出现在一个位置，而会带着整套旁瓣结构；多个点源的旁瓣会互相叠加，强源旁瓣甚至可能掩盖弱源。


In [ ]:
def make_clean_demo_data():
    rng = np.random.default_rng(12)
    n_pix = 801
    l = np.linspace(-0.5, 0.5, n_pix)
    source_l = np.array([l[270], l[575]])
    source_i = np.array([1.0, 0.58])

    positive = np.concatenate([
        rng.normal(0.0, 15.0, 90),
        rng.uniform(8.0, 58.0, 55),
    ])
    positive = np.abs(positive)
    positive = positive[(positive > 0.8) & (positive < 60.0)]
    u = np.sort(np.concatenate([-positive, positive, [0.0]]))

    visibility = np.zeros_like(u, dtype=complex)
    for amp, pos in zip(source_i, source_l):
        visibility += amp * np.exp(-2j * np.pi * u * pos)

    weights = np.ones_like(u)
    weights /= weights.max()
    norm = np.sum(weights)

    phase = np.exp(2j * np.pi * np.outer(l, u))
    dirty = np.real(phase @ (weights * visibility)) / norm

    lag = np.linspace(-1.0, 1.0, 2 * n_pix - 1)
    psf = np.real(np.exp(2j * np.pi * np.outer(lag, u)) @ weights) / norm
    psf /= psf.max()
    dirty /= psf.max()

    return l, lag, u, source_l, source_i, visibility, weights, dirty, psf


def hogbom_clean_1d(dirty, psf, gain=0.08, threshold=0.015, max_iter=900):
    residual = dirty.copy()
    model = np.zeros_like(dirty)
    center = len(psf) // 2
    snapshots = {0: residual.copy()}

    for iteration in range(1, max_iter + 1):
        peak_index = int(np.argmax(np.abs(residual)))
        peak = residual[peak_index]
        if abs(peak) < threshold:
            break
        component = gain * peak
        model[peak_index] += component
        start = center - peak_index
        stop = start + len(dirty)
        residual -= component * psf[start:stop]
        if iteration in (10, 40, 160):
            snapshots[iteration] = residual.copy()

    snapshots[iteration] = residual.copy()
    return model, residual, snapshots, iteration


def estimate_clean_beam_sigma(lag, psf):
    center = len(psf) // 2
    right = center + np.argmax(psf[center:] < 0.5)
    if right == center:
        right = center + 12
    half_width = abs(lag[right] - lag[center])
    fwhm = 2.0 * half_width
    return fwhm / (2.0 * np.sqrt(2.0 * np.log(2.0)))


def restore_model(l, model, residual, sigma):
    restored = np.zeros_like(l)
    indices = np.flatnonzero(np.abs(model) > 1e-5)
    for idx in indices:
        restored += model[idx] * np.exp(-0.5 * ((l - l[idx]) / sigma) ** 2)
    return restored + residual


def true_sky_convolved(l, source_l, source_i, sigma):
    sky = np.zeros_like(l)
    for amp, pos in zip(source_i, source_l):
        sky += amp * np.exp(-0.5 * ((l - pos) / sigma) ** 2)
    return sky


def weighted_psf(lag, u, exponent):
    weights = (np.abs(u) + 1.0) ** exponent
    weights /= weights.max()
    psf = np.real(np.exp(2j * np.pi * np.outer(lag, u)) @ weights) / np.sum(weights)
    psf /= psf.max()
    return weights, psf

fig_dir = Path('figures')
fig_dir.mkdir(exist_ok=True)

l, lag, u, source_l, source_i, visibility, weights, dirty, psf = make_clean_demo_data()
model, residual, snapshots, n_iter = hogbom_clean_1d(dirty, psf)
sigma = estimate_clean_beam_sigma(lag, psf)
restored = restore_model(l, model, residual, sigma)
true_restored = true_sky_convolved(l, source_l, source_i, sigma)

fig, axes = plt.subplots(2, 2, figsize=(12, 7.4), constrained_layout=True)
axes[0, 0].stem(source_l, source_i, basefmt=' ', linefmt='C0-', markerfmt='C0o')
axes[0, 0].set_xlim(-0.5, 0.5)
axes[0, 0].set_ylim(0, 1.15)
axes[0, 0].set_xlabel(r'$l$')
axes[0, 0].set_ylabel('Flux density')
axes[0, 0].set_title('True point-source sky')

axes[0, 1].scatter(u, np.zeros_like(u), s=14, color='C1', alpha=0.75)
axes[0, 1].set_xlim(-62, 62)
axes[0, 1].set_ylim(-0.5, 0.5)
axes[0, 1].set_xlabel(r'$u$')
axes[0, 1].set_yticks([])
axes[0, 1].set_title('Sampled spatial frequencies')

axes[1, 0].plot(lag, psf, color='C2', lw=1.6)
axes[1, 0].axhline(0, color='0.75', lw=0.8)
axes[1, 0].set_xlim(-0.35, 0.35)
axes[1, 0].set_xlabel(r'$\Delta l$')
axes[1, 0].set_ylabel(r'$B_D$')
axes[1, 0].set_title('Dirty beam')

axes[1, 1].plot(l, dirty, color='C3', lw=1.5)
for pos in source_l:
    axes[1, 1].axvline(pos, color='0.35', lw=0.8, ls='--')
axes[1, 1].set_xlim(-0.5, 0.5)
axes[1, 1].set_xlabel(r'$l$')
axes[1, 1].set_ylabel(r'$I_D$')
axes[1, 1].set_title('Dirty image')
fig.savefig(fig_dir / 'clean_1d_dirty_psf.png', dpi=180)
plt.close(fig)

fig, axes = plt.subplots(2, 2, figsize=(12, 7.4), constrained_layout=True)
axes[0, 0].plot(l, dirty, color='0.45', lw=1.1, label='dirty image')
axes[0, 0].stem(l, model, basefmt=' ', linefmt='C0-', markerfmt='C0o', label='CLEAN components')
axes[0, 0].set_xlim(-0.5, 0.5)
axes[0, 0].set_xlabel(r'$l$')
axes[0, 0].set_ylabel('Intensity')
axes[0, 0].set_title('Component model')
axes[0, 0].legend(loc='upper right', frameon=False)

for key, color in zip(sorted(snapshots)[:4], ['0.55', 'C1', 'C2', 'C3']):
    axes[0, 1].plot(l, snapshots[key], lw=1.1, label=f'iter {key}', color=color)
axes[0, 1].set_xlim(-0.5, 0.5)
axes[0, 1].set_xlabel(r'$l$')
axes[0, 1].set_ylabel('Residual')
axes[0, 1].set_title('Residual during minor cycles')
axes[0, 1].legend(loc='upper right', frameon=False)

beam = np.exp(-0.5 * (lag / sigma) ** 2)
beam /= beam.max()
axes[1, 0].plot(lag, psf, color='C2', lw=1.2, label='dirty beam')
axes[1, 0].plot(lag, beam, color='C4', lw=1.8, label='clean beam')
axes[1, 0].set_xlim(-0.18, 0.18)
axes[1, 0].set_xlabel(r'$\Delta l$')
axes[1, 0].set_ylabel('Normalized response')
axes[1, 0].set_title('Main-lobe restoration beam')
axes[1, 0].legend(loc='upper right', frameon=False)

axes[1, 1].plot(l, restored, color='C0', lw=1.5, label='restored image')
axes[1, 1].plot(l, true_restored, color='0.35', lw=1.2, ls='--', label='true sky convolved')
axes[1, 1].plot(l, residual, color='C3', lw=0.9, alpha=0.8, label='residual')
axes[1, 1].set_xlim(-0.5, 0.5)
axes[1, 1].set_xlabel(r'$l$')
axes[1, 1].set_ylabel('Intensity')
axes[1, 1].set_title('Restored image')
axes[1, 1].legend(loc='upper right', frameon=False)
fig.savefig(fig_dir / 'clean_1d_iterations.png', dpi=180)
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.3), constrained_layout=True)
settings = [(-1.1, 'short-baseline emphasis', 'C0'), (0.0, 'equal sampled weights', 'C2'), (1.1, 'long-baseline emphasis', 'C3')]
for exponent, label, color in settings:
    w, b = weighted_psf(lag, u, exponent)
    axes[0].scatter(u, w, s=9, alpha=0.55, color=color, label=label)
    axes[1].plot(lag, b, lw=1.5, color=color, label=label)
axes[0].set_xlim(-62, 62)
axes[0].set_xlabel(r'$u$')
axes[0].set_ylabel('Relative weight')
axes[0].set_title('Visibility weights')
axes[0].legend(loc='upper center', frameon=False, fontsize=9)
axes[1].axhline(0, color='0.75', lw=0.8)
axes[1].set_xlim(-0.24, 0.24)
axes[1].set_xlabel(r'$\Delta l$')
axes[1].set_ylabel('Normalized PSF')
axes[1].set_title('Resulting dirty beams')
axes[1].legend(loc='upper right', frameon=False, fontsize=9)
fig.savefig(fig_dir / 'clean_1d_weighting_psf.png', dpi=180)
plt.close(fig)


图 2.14.1 给出了一维点源天空、有限 `uv` 采样、脏波束和脏图像之间的关系。两个点源本身非常简单；复杂性来自采样函数。脏波束的主瓣给出名义角分辨率，旁瓣则把每个点源的响应复制到其他位置。脏图像中的峰值位置仍然接近真实点源，但峰的形状和周围振荡已经不再只属于源本身。

![一维采样、脏波束和脏图像](figures/clean_1d_dirty_psf.png)

**图 2.14.1** 一维点源天空经过不完整空间频率采样后形成脏波束和脏图像。虚线标出真实点源位置。


### 2.14.3 Högbom CLEAN 的迭代含义<a id='math:sec:hogbom_clean'></a>

Högbom CLEAN 是最经典的 CLEAN 形式。它从残差图像 $R^{(0)}=I_D$ 和空模型 $I_M^{(0)}=0$ 开始。第 $k$ 次迭代时，在残差中找到绝对值最大的像素 $l_k$，记峰值为 $r_k=R^{(k)}(l_k)$。随后把一个强度为 $\gamma r_k$ 的点分量加入模型，并从残差中减去同样强度的移位脏波束：

$$
I_M^{(k+1)}(l)=I_M^{(k)}(l)+\gamma r_k\delta(l-l_k),
$$

$$
R^{(k+1)}(l)=R^{(k)}(l)-\gamma r_k B_D(l-l_k).
$$

这里 $0<\gamma<1$ 称为循环增益。它让算法每一步只移除峰值的一部分，而不是一次性减掉整个脏波束。这样做看似保守，却很重要：当多个源的旁瓣互相叠加时，残差峰值并不一定等于某个真实源的完整通量。较小的循环增益可以降低过减风险，使模型在多次迭代中逐渐收敛。

从线性代数角度看，Högbom CLEAN 不是在显式求 $H^{-1}$，而是在点源字典上做贪婪近似。每一步选择与当前残差最相关的一个移位脏波束，把它对应的点分量加入模型。这与匹配追踪的思想非常接近。由于无线电天空常包含紧致源，且强源旁瓣常是脏图像中最先需要移除的结构，这种简单策略在实际成像中非常有生命力。


图 2.14.2 展示同一个一维例子的 CLEAN 过程。左上图中的竖线是逐步积累的 CLEAN 分量；右上图显示残差峰值随着迭代下降。左下图比较脏波束主瓣与用于恢复的 clean beam；右下图则给出恢复图像。

![一维 Högbom CLEAN 迭代与恢复图像](figures/clean_1d_iterations.png)

**图 2.14.2** Högbom CLEAN 的模型分量、残差演化、clean beam 与恢复图像。恢复图像由 CLEAN 分量与理想化主瓣卷积后再加回残差得到。


### 2.14.4 恢复图像的意义<a id='math:sec:clean_restoration'></a>

CLEAN 迭代得到的 $I_M$ 是一组 delta 分量。它适合表示算法内部的紧致源模型，却不适合作为最终科学图像直接使用。原因有两点。第一，真实干涉阵列的分辨率有限，delta 分量会暗示数据并未真正提供的无限角分辨率；第二，残差图像中仍然保留噪声、未清理弱结构以及模型不完备带来的剩余信号。

因此 CLEAN 通常以恢复图像作为输出：

$$
I_R = B_C * I_M + R,
$$

其中 $B_C$ 是 clean beam，通常取为拟合脏波束主瓣的高斯函数，$R$ 是最终残差图像。这个表达式有明确的物理含义：模型分量以观测可支持的角分辨率呈现，残差则保留数据中没有被点源模型解释的部分。最终图像中的结构既包含模型假设，也包含观测约束；这也是解释 CLEAN 图像时必须同时查看模型、残差和 PSF 的原因。

停止准则同样不能只按迭代次数理解。在无噪声的演示中，可以把阈值设得很低；在真实观测中，继续清理到热噪声以下往往会把噪声峰误认为真实源，产生虚假分量。合理的阈值通常与残差 rms、已知源结构、动态范围要求和校准质量一起决定。


### 2.14.5 主循环、次循环与 Cotton-Schwab 思想<a id='math:sec:cotton_schwab'></a>

Högbom CLEAN 的残差更新完全发生在图像域：找到一个峰值，就从残差图像中减去一个移位脏波束。这一步非常快，所以适合在次循环中反复执行。然而图像域更新依赖一个近似：每个分量的响应都可以由同一个平移不变 PSF 描述。实际成像中会出现网格化误差、非共面基线效应、方向相关效应、频率合成和主波束变化，这些因素都会让“简单移位同一个 PSF”逐渐偏离真实测量方程。

Cotton-Schwab CLEAN 的关键思想是在两层循环之间折中。次循环仍然在图像域快速寻找和减去分量；经过若干次次循环后，主循环回到可见度域，利用当前模型重新预测可见度

$$
V_M = S F I_M,
$$

并计算新的可见度残差

$$
V_R = V_{\rm obs}-V_M,
$$

随后再由

$$
R = F^H S^H W V_R
$$

形成新的残差图像。这样做比每一步都回到可见度域便宜得多，又能周期性地把算法拉回原始测量方程。现代 CLEAN 变体通常都保留了这种“图像域快速次循环 + 可见度域精确主循环”的思想，只是在分量字典、尺度选择、权重处理和并行实现上更加复杂。


### 2.14.6 加权如何改变 PSF<a id='math:sec:clean_weighting'></a>

在正规方程中，权重 $W$ 直接进入 Hessian：

$$
H=F^H S^H W S F.
$$

因此改变权重并不是在成像完成后修饰图像，而是在改变反问题本身。短基线权重大时，脏波束通常更宽，对大尺度结构更敏感，点源分辨率较低；长基线权重大时，主瓣变窄，名义分辨率提高，但旁瓣和噪声行为也可能更难处理。自然加权、均匀加权和 Briggs 鲁棒加权都是在这条权衡曲线上选择不同位置。

图 2.14.3 用同一组一维采样点展示权重对 PSF 的影响。这里的权重族只是教学示意，不等同于实际软件中的完整实现；它足以说明一个原则：PSF 是采样与权重共同决定的，去卷积面对的“脏波束”不是阵列几何的唯一结果，而是观测设计、数据标定、权重选择共同塑造的对象。

![一维权重与脏波束](figures/clean_1d_weighting_psf.png)

**图 2.14.3** 改变空间频率权重会同时改变主瓣宽度和旁瓣结构。强调长基线通常提高名义分辨率，强调短基线通常提高对平滑结构的敏感性。


### 2.14.7 CLEAN 的适用边界<a id='math:sec:clean_limits'></a>

CLEAN 的成功来自一个相当具体的先验：天空可以由若干紧致分量逐步逼近。这个先验在点源、喷流结、致密核和许多高对比度场景中非常有效；面对弥散射电晕、超新星遗迹、银河系大尺度辐射或低信噪比扩展结构时，单尺度点源 CLEAN 往往会把连续结构分解成许多小点，残差也可能带有明显的相关纹理。多尺度 CLEAN、最大熵方法、压缩感知方法和贝叶斯成像方法，都是在尝试用更合适的先验描述不同类型的天空。

本节的一维推导也刻意忽略了许多真实问题：二维 `uv` 覆盖、主波束、宽带频率合成、$w$ 项、方向相关增益、极化泄漏和校准误差都没有进入模型。忽略这些因素不是因为它们不重要，而是因为 CLEAN 的核心数学关系已经能在最小模型中看清楚：有限采样给出脏波束，脏图像是真天空与脏波束的卷积，CLEAN 通过逐步建立模型并更新残差来近似求解病态反问题。


### 2.14.8 本节要点

一维 CLEAN 演示把第 2 章的几条数学主线连接在一起。采样函数使傅里叶空间信息不完整；加权反变换得到的脏图像等于真天空与脏波束卷积；正规方程中的 Hessian 正是描述这种模糊的线性算子。由于 Hessian 通常病态，直接求逆并不可行。Högbom CLEAN 以点源稀疏模型作为先验，在残差中反复寻找最强分量并减去移位 PSF；恢复图像则用 clean beam 给模型赋予观测可支持的角分辨率，并把残差加回。Cotton-Schwab CLEAN 进一步把快速图像域次循环与可见度域主循环结合起来，为实际二维成像算法奠定了基本框架。


***

* 下一节：[2.x 延伸阅读与参考文献](2_x_further_reading_and_references.ipynb)
